<a href="https://colab.research.google.com/github/Hania-Emaan/code-switching-codesaviours-si26-Hania-Emaan/blob/main/SI26_Week7_Hania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1 — Load dataset and prepare for training
!pip install transformers torch datasets seqeval

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Exact path from your Google Drive screenshot
file_path = '/content/drive/MyDrive/CodeSaviours_Project2/dataset.csv'
df = pd.read_csv(file_path)

# Create label mapping
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

# Group by sentence (handles 'sentence' or 'sentence_id', and 'word' or 'tokens')
sentence_col = 'sentence' if 'sentence' in df.columns else 'sentence_id'
word_col = 'word' if 'word' in df.columns else 'tokens'
label_col = 'label' if 'label' in df.columns else 'labels'

sentences = df.groupby(sentence_col).apply(
    lambda x: {'words': x[word_col].tolist(), 'labels': x[label_col].tolist()}
).tolist()

# Split into train (80%) and test (20%)
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

print(f'Total sentences: {len(sentences)}')
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=3d3162713f2ae04672bcfcbfa1195f6868b607d950f9f8548387e4593b5999dd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
Total sentences: 152
Training sentences: 121
Testing sentences: 31


/tmp/ipykernel_2370/3578438674.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby(sentence_col).apply(


In [7]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from seqeval.metrics import classification_report
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                        TrainingArguments, Trainer, DataCollatorForTokenClassification)
from huggingface_hub import notebook_login

# 1. Load Dataset from Google Drive
file_path = '/content/drive/MyDrive/CodeSaviours_Project2/dataset.csv'
df = pd.read_csv(file_path)

label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

sentence_col = 'sentence' if 'sentence' in df.columns else 'sentence_id'
word_col = 'word' if 'word' in df.columns else 'tokens'
label_col = 'label' if 'label' in df.columns else 'labels'

sentences = df.groupby(sentence_col, group_keys=False).apply(
    lambda x: {'words': x[word_col].tolist(), 'labels': x[label_col].tolist()},
    include_groups=False
).tolist()

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

# 2. Tokenizer & Alignment
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_dataset = Dataset.from_dict({
    "words": [item["words"] for item in train_data],
    "labels": [item["labels"] for item in train_data]
}).map(tokenize_and_align_labels, batched=True)

test_dataset = Dataset.from_dict({
    "words": [item["words"] for item in test_data],
    "labels": [item["labels"] for item in test_data]
}).map(tokenize_and_align_labels, batched=True)

# 3. Model & Trainer (Updated processing_class parameter)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

# 4. Train
print("Starting training...")
trainer.train()
print("Training complete!")

# 5. Evaluate and print F1 scores for submission
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

print("\n--- EVALUATION REPORT ---")
report = classification_report(true_labels, true_predictions)
print(report)

Map:   0%|          | 0/121 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!



--- EVALUATION REPORT ---
              precision    recall  f1-score   support

          IX       0.00      0.00      0.00         1
          NG       0.99      0.99      0.99       205
          RD       0.91      0.89      0.90        36

   micro avg       0.98      0.97      0.97       242
   macro avg       0.63      0.63      0.63       242
weighted avg       0.97      0.97      0.97       242



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [10]:
from transformers import pipeline

# Load pipeline using your local model and tokenizer
nlp_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Test sentence
text = "I am tired lekin phir b kaam to done krna e ha"

# Run inference
predictions = nlp_pipeline(text)

print("\n--- Predictions ---")
for pred in predictions:
    print(f"Entity: {pred['entity_group']} | Text: '{pred['word']}' | Score: {pred['score']:.4f}")


--- Predictions ---
Entity: ENG | Text: 'I am tired' | Score: 0.9435
Entity: URD | Text: 'lekin phir b kaam to' | Score: 0.9140
Entity: ENG | Text: 'done' | Score: 0.9679
Entity: URD | Text: 'krna e ha' | Score: 0.9804


In [11]:
from huggingface_hub import notebook_login

# 1. Login to Hugging Face
notebook_login()

# 2. Define repository name
repo_name = 'code-switching-codesaviours-si26-hania'

# 3. Push model and tokenizer
print("Pushing model to Hugging Face...")
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("Published successfully!")

Pushing model to Hugging Face...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_dlxy9/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpdk3lp278/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Published successfully!
